### Задачи к Лекции 4

__Исходные данные__ 

Дан файл **"mlbootcamp5_train.csv"**. В нем содержатся данные об опросе 70000 пациентов с целью определения наличия заболеваний сердечно-сосудистой системы (ССЗ). Данные в файле промаркированы и если у человека имееются ССЗ, то значение **cardio** будет равно 1, в противном случае - 0. Описание и значения полей представлены во второй лекции.

__Загрузка файла__

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from matplotlib import pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [10, 5]


df = pd.read_csv("C:/Users/akush/Downloads/Telegram Desktop/mlbootcamp5_train.csv", 
                 sep=";", 
                 index_col="id")
# Делаем one-hot кодирование
chol = pd.get_dummies(df["cholesterol"], prefix="chol")
gluc = pd.get_dummies(df["gluc"], prefix="gluc")
df = pd.concat([df, chol, gluc], axis=1)

# Делаем пол бинарным признаком
df["gender_bin"] = df["gender"].map({1: 0, 2: 1})
df.head()







,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,chol_1,chol_2,chol_3,gluc_1,gluc_2,gluc_3,gender_bin
id,,,,,,,,,,,,,,,,,,,
0,18393,2,168,62.0,110,80,1,1,0,0,1,0,1,0,0,1,0,0,1
1,20228,1,156,85.0,140,90,3,1,0,0,1,1,0,0,1,1,0,0,0
2,18857,1,165,64.0,130,70,3,1,0,0,0,1,0,0,1,1,0,0,0
3,17623,2,169,82.0,150,100,1,1,0,0,1,1,1,0,0,1,0,0,1
4,17474,1,156,56.0,100,60,1,1,0,0,0,0,1,0,0,1,0,0,0


In [2]:
data = df[(df["ap_hi"] >= 50) & (df["ap_hi"] <= 200)]

## Задачи

__1. Хоть в sklearn и присутствует реализация метода k-ближайших соседей, я же предлагаю попробовать вам написать его самостоятельно.__

* __создать классификатор используя только pandas, numpy и scipy. Гиперпараметром данного классификатора должно быть число ближайших соседей. (Необязательно) можно добавить метрику расстояния и выбор весов.__
* __С помощью кросс-валидации найти оптимальное количество ближайших соседей и (необязательно) набор признаков.__

Алгоритм работы классификатора:
 1. Для заданного прецедент  $\vec{x}$ мы считаем расстояние до всех прецедентов в обучающей выборке.
 2. Сортируем прецеденты по расстоянию до $\vec{x}$.
 3. Отбираем $k$ минимальных значений
 4. Устраиваем голосование между отобранными прецедент.

In [3]:
data = data[['age', 'weight', 'height', 'ap_lo', 'ap_hi','cardio']]
data.head()


,age,weight,height,ap_lo,ap_hi,cardio
id,,,,,,
0,18393,62.0,168,80,110,0
1,20228,85.0,156,90,140,1
2,18857,64.0,165,70,130,1
3,17623,82.0,169,100,150,1
4,17474,56.0,156,60,100,0


NameError: name 'data_norm' is not defined

In [4]:
# рассчет расстояний без весов и метрики расстояний
# k - число соседей
# el - признаки элемента для сравнения и предсказания
# data - прочие элементы, классифицированные и их признаки в датафрейме
def KNN(k, el, data):
    # считаем расстояние до всех элементов
    d = []
    data_norm = data[['age', 'weight', 'height', 'ap_lo', 'ap_hi']]
    for i in range(0, len(data_norm)):
        d.append(np.sqrt(np.square(data_norm.iloc[el] - data_norm.iloc[i]).sum()))
    data['distance'] = d
    # сортируем от ближнего к дальнему
    data = data.sort_values(by=['distance'])
    # выбираем К ближайших
    KNN = data[1:k+1]
    # делаем выбор 
    if 1 in KNN['cardio'].value_counts().index:
        if KNN['cardio'].value_counts()[1] > KNN['cardio'].value_counts().get(0, 0):
            predict = 1
        else:
            predict = 0
    else:
        predict = 0
    return predict

In [5]:
# рассчет расстояний с весами и метрикой расстояний
def KNN_weight(k, el, data):
    # считаем расстояние до всех элементов
    d = []
   #вес соседа, где 1 - весь самого ближнего из выбранных К соседей
    data_norm = data[['age', 'weight', 'height', 'ap_lo', 'ap_hi']]
    for i in range(0, len(data_norm)):
        d.append(np.sqrt(np.square(data_norm.iloc[el] - data_norm.iloc[i]).sum()))
    data['distance'] = d
    # сортируем от ближнего к дальнему
    data = data.sort_values(by=['distance'])
    # выбираем К ближайших
    KNN = data[1:k+2]
    KNN['weightP'] = np.zeros(len(KNN))
    #рассчитываем вес соседа, где 1 - весь самого ближнего из выбранных К соседей, далее считаем вес остальных относительного ближнего
    for i in range(len(KNN)): KNN['weightP'].iloc[i] = KNN['distance'].iloc[0]/KNN['distance'].iloc[i]
    # делаем выбор той категории (1 или 0) у кого больше веса 
        
    if 1 in KNN['cardio'].value_counts().index:
        if KNN.groupby(['cardio'])['weightP'].sum()[1] > KNN.groupby(['cardio'])['weightP'].sum()[0]: 
            predict = 1
        else: predict = 0
    else:
        predict = 0
    return predict

In [15]:
predict = []
predict_w = []
data_norm = data[['age', 'weight', 'height', 'ap_lo', 'ap_hi']]
data_norm = (data_norm-data_norm.min())/(data_norm.max()-data_norm.min())
data_norm['cardio'] = data['cardio']
for i in range(0, 20):
    a = KNN(5, i, data_norm)
    b = KNN_weight(5, i, data_norm)
    predict.append(a)
    predict_w.append(b)
    print(i, "=", a)

#print(predict)

0 = 0
1 = 1
2 = 0
3 = 1
4 = 0
5 = 1
6 = 1
7 = 1
8 = 0
9 = 0
10 = 0
11 = 0
12 = 0
13 = 0
14 = 1
15 = 1
16 = 1
17 = 0
18 = 0
19 = 1


In [16]:
#провереям точность
target = data['cardio'][:20]
print(20,
    (target == predict).sum() / 20)

20 0.7


In [17]:
print(20,
    (target == predict_w).sum()/20)

20 0.75


**Комментарии:** Провести анализ точности пришлось на 20 примерах, т.к. код выполняется очень долгое время. На их основе точность получилась 0,7 для невзвешенного алгоритма и 0,75 для взвешенного

  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.64.0
    Uninstalling tqdm-4.64.0:
      Successfully uninstalled tqdm-4.64.0
  Attempting uninstall: requests
    Found existing installation: requests 2.27.1
    Uninstalling requests-2.27.1:
      Successfully uninstalled requests-2.27.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.4 requires pathlib, which is not installed.
anaconda-project 0.10.2 requires ruamel-yaml, which is not installed.


**2. Определить какой из трех классификаторов (kNN, наивный Байес, решающее дерево) лучший в каждой метрике по отдельности: accuracy, F1-мера, ROC AUC, функция потерь. Использовать набор признаков: 'age', 'weight', 'height', 'ap_lo', 'ap_hi'.**

**(Необязательно) Найти оптимальный набор признаков.**

In [6]:
# Your code here

**Комментарии:** Ваши комментарии здесь.